### 데이터 호출

In [1]:
import pandas as pd

In [2]:
# 데이터 호출
data = pd.read_csv('경로/파일명.csv', encoding = 'cp949')
data

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_31476\4231032148.py:2: DtypeWarning: Columns (0,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('경로/파일명.csv', encoding = 'cp949')


,통계연도,구명,동명,성별명,나이대명,인구수,행정동코드,나이대코드
0,stats_yr,sigungu_nm,epmndn_nm,sex_nm,agegrd_nm,popl_cnt,admdong_cd,agegrd_cd
1,2023,종로구,가회동,계,계,3860,1111060000,-1
2,2023,종로구,가회동,여자,85 - 89세,46,1111060000,90
3,2023,종로구,혜화동,계,계,15869,1111065000,-1
4,2023,종로구,혜화동,여자,60 - 64세,499,1111065000,65
...,...,...,...,...,...,...,...,...
446024,2025,중랑구,중화2동,여자,80 - 84세,459,1126061000,85
446025,2025,중랑구,중화2동,여자,85 - 89세,239,1126061000,90
446026,2025,중랑구,중화2동,여자,90 - 94세,91,1126061000,95
446027,2025,중랑구,중화2동,여자,95 - 99세,17,1126061000,100


### 필요한 컬럼 추출 및 삭제

In [50]:
df = data.copy()

# 첫번째 행 필요없으니까 삭제
df = df.iloc[1:]

# 필요한 컬럼만 추출 → 통계연도, 구명, 동명, 성별명, 나이대명, 인구수
df = df[['통계연도', '구명', '동명', '성별명', '나이대명', '인구수']]

# 최근 인구 통계를 볼 계획이니까 2025년 데이터만 선택
df = df[df['통계연도'] == 2025]

# 남녀 구분해야 하므로 '성별명' 집계 된 데이터 제외
df = df[df['성별명'] != '계']

# 나이대도 분류할 거니까 '나이대명' 집계 된 데이터 제외
df = df[df['나이대명'] != '계'].reset_index(drop = True) # 데이터를 깔끔하게 보기 위해 기존 인덱스를 삭제하면서 리셋

# 컬럼명 정리
df.columns = ['년도', '자치구', '동', '성별', '연령대', '인구수']

### 나이대, 국민건강조사 틀에 맞게 수정하기 ```5세 단위 → 10세 단위```

In [37]:
# 나이대 구성 확인
df['연령대'].drop_duplicates()

0       0 - 4세
1     10 - 14세
2         100+
3     15 - 19세
4     20 - 24세
5     25 - 29세
6     30 - 34세
7     35 - 39세
8     40 - 44세
9     45 - 49세
10      5 - 9세
11    50 - 54세
12    55 - 59세
13    60 - 64세
14    65 - 69세
15    70 - 74세
16    75 - 79세
17    80 - 84세
18    85 - 89세
19    90 - 94세
20    95 - 99세
Name: 연령대, dtype: object

In [51]:
# 10세 미만은 필요 없으므로 제거
df = df[(df['연령대'] != '0 - 4세') & (df['연령대'] != '5 - 9세')]

# 나이대 그룹핑
age_group = {
    '10 - 14세': '10대', 
    '15 - 19세': '10대',
    '20 - 24세': '20대', 
    '25 - 29세': '20대',
    '30 - 34세': '30대', 
    '35 - 39세': '30대',
    '40 - 44세': '40대', 
    '45 - 49세': '40대',
    '50 - 54세': '50대', 
    '55 - 59세': '50대',
    '60 - 64세': '60대', 
    '65 - 69세': '60대',
    '70 - 74세': '70세 이상', 
    '75 - 79세': '70세 이상',
    '80 - 84세': '70세 이상', 
    '85 - 89세': '70세 이상',
    '90 - 94세': '70세 이상', 
    '95 - 99세': '70세 이상',
    '100+': '70세 이상'
}

# 연령대 수정
df['연령대'] = df['연령대'].map(age_group)

# 동일한 연령대끼리 인구 수 합산하기
df = df.groupby(['년도', '자치구', '동', '성별', '연령대'])['인구수'].sum().reset_index()

In [53]:
df.to_csv('경로/파일명.csv', index = False, encoding = 'cp949')